# SPINE-GPE v7 — PNADc Certifier v1.2.0

Este notebook executa apenas a camada de golden tests SIDRA sobre os Parquets já produzidos. Ele não relê os TXT fixed-width.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7")
SCRIPT_NAME = "SPINE_GPEv7_PNADC_CERTIFIER_v1.2.0.py"

candidates = [
    Path("/content") / SCRIPT_NAME,
    ROOT / "scripts" / SCRIPT_NAME,
]

SCRIPT = next((path for path in candidates if path.exists()), None)

print("ROOT:", ROOT)
print("SCRIPT:", SCRIPT)

if not ROOT.exists():
    raise FileNotFoundError(f"Raiz não encontrada: {ROOT}")
if SCRIPT is None:
    raise FileNotFoundError(
        "Envie o script para /content ou coloque-o em SPINE-GPEv7/scripts/."
    )


In [ ]:
import importlib
import subprocess
import sys

packages = {
    "numpy": "numpy>=1.26",
    "pandas": "pandas>=2.2",
    "pyarrow": "pyarrow>=17",
    "scipy": "scipy>=1.12",
    "requests": "requests>=2.31",
    "urllib3": "urllib3>=2.2",
    "bs4": "beautifulsoup4>=4.12",
    "openpyxl": "openpyxl>=3.1",
    "xlrd": "xlrd>=2.0",
    "lxml": "lxml>=5",
}

missing = []
for module_name, package_name in packages.items():
    try:
        importlib.import_module(module_name)
        print("OK:", module_name)
    except ImportError:
        missing.append(package_name)

if missing:
    print("Instalando:", missing)
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-cache-dir",
            "--prefer-binary",
            *missing,
        ],
        check=True,
    )

print("Ambiente pronto.")


In [ ]:
import json
import py_compile

py_compile.compile(str(SCRIPT), doraise=True)
print("py_compile: OK")

phase0_lock = ROOT / "00_admin" / "PHASE0_LOCK.json"
phase0 = json.loads(phase0_lock.read_text(encoding="utf-8"))
print("PHASE0:", phase0.get("status"))

if phase0.get("status") != "RELEASED":
    raise RuntimeError("A Fase 0 estrutural não está RELEASED.")


In [ ]:
required = [
    ROOT / "03_processed/10_pnadc_certified/certified_pnadc_platform_2022.parquet",
    ROOT / "03_processed/10_pnadc_certified/certified_pnadc_platform_2024.parquet",
    ROOT / "00_admin/registry/certified_pnadc_platform_2022_manifest.json",
    ROOT / "00_admin/registry/certified_pnadc_platform_2024_manifest.json",
]

snapshots = ROOT / "02_interim/10_pnadc_certification/sidra_snapshots"
required_tables = [9432, 9441, 9442, 9443, 9642]

for path in required:
    print(path.exists(), path)
    if not path.exists():
        raise FileNotFoundError(path)

for table in required_tables:
    matches = list(snapshots.glob(f"sidra_{table}_*.csv"))
    print(f"SIDRA {table}:", matches)
    if not matches:
        raise FileNotFoundError(f"Snapshot SIDRA {table} ausente")

print("SIDRA 9518 opcional:", list(snapshots.glob("sidra_9518_*.csv")))


In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    str(SCRIPT),
    "--root",
    str(ROOT),
    "--mode",
    "sidra-only",
    "--sidra-source",
    "cache",
    "--strict",
]

print(" ".join(cmd))

result = subprocess.run(
    cmd,
    text=True,
    capture_output=True,
    check=False,
)

print("STDOUT:")
print(result.stdout)
print("STDERR:")
print(result.stderr)
print("Exit code:", result.returncode)

if result.returncode not in {0}:
    raise RuntimeError(
        "Execução bloqueada. Consulte critical_failures no lock."
    )


In [ ]:
import json

lock_path = ROOT / "00_admin" / "PNADC_CERTIFICATION_LOCK.json"
lock = json.loads(lock_path.read_text(encoding="utf-8"))

print(json.dumps(lock, ensure_ascii=False, indent=2))

status = lock.get("status")
print("STATUS:", status)

if status == "BLOCKED":
    raise RuntimeError("Há gate crítico bloqueado.")

if status == "CORE_CERTIFIED":
    print(
        "Núcleo PNADc direto certificado. "
        "Rendimento real e/ou informalidade permanecem pendências secundárias."
    )
elif status == "CERTIFIED":
    print("PNADc direta integralmente certificada.")


In [ ]:
from pathlib import Path

report_path = Path(lock["report"])
print("Relatório:", report_path)

report = report_path.read_text(encoding="utf-8")
print(report[:12000])
